In [33]:
# 1. Import Libraries and Download NLTK resources
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download NLTK resourc
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

True

In [34]:
df = pd.read_csv(r"C:\Users\pc\OneDrive\Music\Desktop\ML_Projects\Dataset\emails.csv")
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5728 entries, 0 to 5727
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    5728 non-null   object
 1   spam    5728 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 89.6+ KB


In [36]:
ps = PorterStemmer()
stop_words = set(stopwords.words("english"))

def clean_data(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\r\n", " ", text)
    text = re.sub(r"[^A-Za-z0-9\s]", " ", text)  # Remove Punctuation
    text = re.sub(r"http\S+", " ", text)         # Remove URLS
    text = re.sub(r"[<.*?>]", " ", text)         # Remove HTML tags
    text = text.strip().lower()                  # Convert into lowercase
    
    # Tokenize, remove stopwords, and apply stemming
    tokens = word_tokenize(text)
    cleaned_tokens = [ps.stem(word) for word in tokens if word not in stop_words]
    return " ".join(cleaned_tokens)

print("Cleaning text data...")
df["text"] = df["text"].apply(clean_data)
df.head()

Original Shape: (5728, 2)
Shape after removing duplicates: (5695, 2)


,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [37]:
# 4. Feature Extraction (TF-IDF Vectorization) & Train-Test Split
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df["text"]).toarray()
y = df["spam"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (4556, 5000)
X_test shape: (1139, 5000)
